# 🚀 Kaggle BGE Reranker Server
This notebook loads the **BAAI/bge-reranker-large** cross-encoder model and serves it as a FastAPI relevance scoring backend. It exposes the API to the public internet using a secure `pyngrok` tunnel.

### ⚙️ Instructions
1. Turn on the **GPU Accelerator** in the notebook settings (select **GPU T4 x2** or **GPU T4 x1**).
2. Add your **Ngrok Auth Token** to Kaggle Secrets:
   - Go to **Add-ons -> Secrets** in the top menu.
   - Add a secret named `NGROK_AUTH_TOKEN` with your token as the value.
   - Make sure the checkbox next to the secret is checked to share it with the notebook.
3. Run all cells in order.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    torch \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    pydantic

## 🔑 Step 2: Load Secrets and Import Libraries

In [ ]:
import os
import torch
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "BAAI/bge-reranker-large"
PORT = 8000

# Fetch secret token from Kaggle user secrets
NGROK_AUTH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.")
except Exception as e:
    print("❌ Failed to load secret from Kaggle User Secrets. Checking environment variables...")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN is not set. Please add it to Add-ons -> Secrets.")

## 🤖 Step 3: Load Reranker Model

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading cross-encoder model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"✅ Reranker model loaded successfully onto: {model.device}")

## 🛠️ Step 4: Define API Schemas and Router

In [ ]:
app = FastAPI(title="Kaggle BGE Reranker API")

class RerankRequest(BaseModel):
    query: str
    documents: list[str]

class RerankResultItem(BaseModel):
    index: int
    score: float

class RerankResponse(BaseModel):
    success: bool
    results: list[RerankResultItem]
    error_message: str

def compute_scores_internal(query: str, documents: list[str]) -> list[dict]:
    pairs = [[query, doc] for doc in documents]
    
    features = tokenizer(
        pairs,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**features)
        scores = outputs.logits.view(-1).float().cpu().tolist()
        
    results = [
        {"index": idx, "score": float(score)}
        for idx, score in enumerate(scores)

    ]
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME
    }

@app.post("/rerank")
def rerank(req: RerankRequest):
    try:
        results = compute_scores_internal(req.query, req.documents)
        return RerankResponse(
            success=True,
            results=[RerankResultItem(**item) for item in results],
            error_message=""
        )
    except Exception as e:
        return RerankResponse(
            success=False,
            results=[],
            error_message=str(e)
        )

## 🚀 Step 5: Start Public Tunnel and FastAPI Server

In [ ]:
# Apply nesting patch to allow uvicorn inside Jupyter event loop
nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(PORT)
    print("\n" + "="*80)
    print(f"🚀 SERVER PUBLIC URL: {public_url}")
    print("Copy the URL above and paste it into your Backed/.env as RERANKER_SERVER_URL")
    print("="*80 + "\n")
else:
    print("\n❌ Unable to set up ngrok tunnel without an authentication token.")

# Launch server directly in the notebook's active event loop
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()